In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/AB_NYC_2019.csv
/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/New_York_City_.png


In [2]:
airbnb=pd.read_csv("/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/AB_NYC_2019.csv")
#sample rows
airbnb.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [3]:
#print the null values
print(airbnb.isnull().sum())

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64


In [4]:
#get the summary 
# airbnb.describe()
print(airbnb.describe())

                 id       host_id      latitude     longitude         price  \
count  4.889500e+04  4.889500e+04  48895.000000  48895.000000  48895.000000   
mean   1.901714e+07  6.762001e+07     40.728949    -73.952170    152.720687   
std    1.098311e+07  7.861097e+07      0.054530      0.046157    240.154170   
min    2.539000e+03  2.438000e+03     40.499790    -74.244420      0.000000   
25%    9.471945e+06  7.822033e+06     40.690100    -73.983070     69.000000   
50%    1.967728e+07  3.079382e+07     40.723070    -73.955680    106.000000   
75%    2.915218e+07  1.074344e+08     40.763115    -73.936275    175.000000   
max    3.648724e+07  2.743213e+08     40.913060    -73.712990  10000.000000   

       minimum_nights  number_of_reviews  reviews_per_month  \
count    48895.000000       48895.000000       38843.000000   
mean         7.029962          23.274466           1.373221   
std         20.510550          44.550582           1.680442   
min          1.000000           0.00

In [5]:
#  use the filtered dataframe
data = airbnb[airbnb['price'] > 0].copy()

# cap price and minimum_nights at their 99th percentile
price_cap = data['price'].quantile(0.99)
nights_cap = data['minimum_nights'].quantile(0.99)

# create two columns additional for the data 
data['price_capped'] = data['price'].clip(upper=price_cap)
data['minimum_nights_capped'] = data['minimum_nights'].clip(upper=nights_cap)

# sanity check — did it work?
print(data['price_capped'].max())
print(data['minimum_nights_capped'].max())

799
45


In [6]:
# fill reviews_per_month nulls with 0  no reviews means literally zero reviews per month
data['reviews_per_month'] = data['reviews_per_month'].fillna(0)

# sanity check
print(data['reviews_per_month'].isnull().sum())

0


In [7]:
# days we had reveiws 
data['has_reviews'] = data['number_of_reviews'] > 0
print(data['has_reviews'].value_counts())

has_reviews
True     38833
False    10051
Name: count, dtype: int64


In [8]:
# convert availability_365 (days) into months available
data['months_available'] = data['availability_365'] / 30
data['demand_density'] = data['reviews_per_month'] / data['months_available'].replace(0, np.nan)

# summary of the demand desnsity figure 
print(data['demand_density'].describe())

count    31354.000000
mean         1.668995
std          7.861610
min          0.000000
25%          0.027124
50%          0.205263
75%          0.744000
max        305.100000
Name: demand_density, dtype: float64
